## Load data and remove non-feature columns

The same joined query from the EDA notebook pulls all 21 columns from
Postgres. `customer_id` is an identifier, not a predictive feature, so it's
dropped here before anything else happens.

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

query = """
SELECT
    c.customer_id, c.gender, c.senior_citizen, c.partner, c.dependents, c.tenure,
    s.phone_service, s.multiple_lines, s.internet_service, s.online_security,
    s.online_backup, s.device_protection, s.tech_support, s.streaming_tv, s.streaming_movies,
    b.contract, b.paperless_billing, b.payment_method, b.monthly_charges, b.total_charges,
    ch.churn
FROM customers c
JOIN services s ON c.customer_id = s.customer_id
JOIN billing b ON c.customer_id = b.customer_id
JOIN churn_status ch ON c.customer_id = ch.customer_id;
"""

df = pd.read_sql(query, engine)
df = df.drop(columns=["customer_id"])
print(df.shape)
df.head()

(7043, 20)


,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,Female,False,True,False,1,False,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,False
1,Male,False,False,False,34,True,No,DSL,Yes,No,Yes,No,No,No,One year,False,Mailed check,56.95,1889.50,False
2,Male,False,False,False,2,True,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,True
3,Male,False,False,False,45,False,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,False
4,Female,False,False,False,2,True,No,Fiber optic,No,No,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,True


In [2]:
X = df.drop(columns=["churn"])
y = df["churn"].astype(int)  # True/False -> 1/0

print(X.shape, y.shape)
print(y.value_counts(normalize=True) * 100)

(7043, 19) (7043,)
churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())

Train shape: (5634, 19)
Test shape: (1409, 19)

Train churn rate: 0.2653532126375577
Test churn rate: 0.2654364797728886


- `test_size=0.2` — standard 80/20 split, 5,634 training rows and 1,409 test rows
- `stratify=y` — this is the one that matters most. Without it, a random split could accidentally put 30% churn in train and 20% in test (or any other mismatch), especially risky on an already-imbalanced target. stratify=y forces both splits to preserve the same ~26.5% churn ratio.
- `random_state=42` — makes the split reproducible; anyone re-running your notebook gets the identical split

In [4]:
# Identify column types
binary_cols = ["senior_citizen", "partner", "dependents", "phone_service", "paperless_billing"]
# already 0/1 from the CSV load step — no encoding needed, just confirm dtype

multi_cat_cols = [
    "gender", "multiple_lines", "internet_service", "online_security",
    "online_backup", "device_protection", "tech_support", "streaming_tv",
    "streaming_movies", "contract", "payment_method"
]

print(X_train[binary_cols].dtypes)

senior_citizen       bool
partner              bool
dependents           bool
phone_service        bool
paperless_billing    bool
dtype: object


In [5]:
# One-hot encode the multi-category columns
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

encoder.fit(X_train[multi_cat_cols])

X_train_encoded = pd.DataFrame(
    encoder.transform(X_train[multi_cat_cols]),
    columns=encoder.get_feature_names_out(multi_cat_cols),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    encoder.transform(X_test[multi_cat_cols]),
    columns=encoder.get_feature_names_out(multi_cat_cols),
    index=X_test.index
)

print(X_train_encoded.shape, X_test_encoded.shape)
X_train_encoded.head()

(5634, 22) (1409, 22)


,gender_Male,multiple_lines_No phone service,multiple_lines_Yes,internet_service_Fiber optic,internet_service_No,online_security_No internet service,online_security_Yes,online_backup_No internet service,online_backup_Yes,device_protection_No internet service,...,tech_support_Yes,streaming_tv_No internet service,streaming_tv_Yes,streaming_movies_No internet service,streaming_movies_Yes,contract_One year,contract_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check
3738,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3151,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4861,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3867,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
3811,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


## Encoding categorical features

Binary Yes/No-style columns are already 0/1 from the data-loading step and
need no further work. The remaining multi-category columns (contract,
internet service, payment method, and the service add-ons) are one-hot
encoded with `OneHotEncoder`.

The encoder is fit only on the training set, then used to transform both
train and test — the same leakage-safety rule as the split above.
`drop="first"` avoids redundant columns, and `handle_unknown="ignore"`
protects against any category that might only appear in the test set.

The "No internet service" category flagged during EDA needed no special
handling — one-hot encoding naturally gives it its own column, correctly
separating that group from a real "No."

## Rebuilding the full feature table

The binary columns, the three numeric columns (`tenure`, `monthly_charges`,
`total_charges`), and the newly one-hot-encoded columns are combined back
into one table. `total_charges` is kept despite its strong correlation
with `tenure` (0.83, found in EDA) — tree-based models handle correlated
features reasonably well, and its actual contribution will be checked with
SHAP in the modelling stage rather than assumed away here.

Indexes are reset before concatenating to avoid any silent row
misalignment between the original and encoded columns.

In [6]:
numeric_cols = ["tenure", "monthly_charges", "total_charges"]

X_train_final = pd.concat(
    [X_train[binary_cols + numeric_cols].reset_index(drop=True),
     X_train_encoded.reset_index(drop=True)],
    axis=1
)

X_test_final = pd.concat(
    [X_test[binary_cols + numeric_cols].reset_index(drop=True),
     X_test_encoded.reset_index(drop=True)],
    axis=1
)
X_train_final[binary_cols] = X_train_final[binary_cols].astype(int)
X_test_final[binary_cols] = X_test_final[binary_cols].astype(int)

print(X_train_final.shape, X_test_final.shape)
X_train_final.head()

(5634, 30) (1409, 30)


,senior_citizen,partner,dependents,phone_service,paperless_billing,tenure,monthly_charges,total_charges,gender_Male,multiple_lines_No phone service,...,tech_support_Yes,streaming_tv_No internet service,streaming_tv_Yes,streaming_movies_No internet service,streaming_movies_Yes,contract_One year,contract_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check
0,0,0,0,1,1,8,20.20,140.95,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0,0,0,1,0,47,75.45,3545.10,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0,0,0,0,0,20,24.90,505.95,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0,1,1,1,0,19,25.60,485.90,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0,0,0,1,0,50,20.15,989.05,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


In [7]:
X_train_final.dtypes

senior_citizen                              int32
partner                                     int32
dependents                                  int32
phone_service                               int32
paperless_billing                           int32
tenure                                      int64
monthly_charges                           float64
total_charges                             float64
gender_Male                               float64
multiple_lines_No phone service           float64
multiple_lines_Yes                        float64
internet_service_Fiber optic              float64
internet_service_No                       float64
online_security_No internet service       float64
online_security_Yes                       float64
online_backup_No internet service         float64
online_backup_Yes                         float64
device_protection_No internet service     float64
device_protection_Yes                     float64
tech_support_No internet service          float64


In [11]:
from imblearn.over_sampling import SMOTENC

# find which columns indexes are cataogorical

numeric_cols = ["tenure" , "monthly_charges" , "total_charges"]
cat_col_indices=[ i for i, col in enumerate(X_train_final.columns) if col not in numeric_cols]
print("Categorical column count:", len(cat_col_indices))
print("Total columns:", X_train_final.shape[1])

Categorical column count: 27
Total columns: 30


## Handling class imbalance: SMOTENC

Standard SMOTE interpolates between neighboring points to create synthetic
minority-class rows, which is meaningless for one-hot encoded categorical
columns (interpolating between 0 and 1 produces an invalid fractional
category). SMOTENC fixes this: it interpolates the numeric columns
normally, but takes the mode of categorical columns among nearest
neighbors instead, so every synthetic row stays valid.

This is applied only to the training set, never the test set, since the
test set must stay representative of real, unaltered data.

In [14]:
smote_nc = SMOTENC(categorical_features = cat_col_indices , random_state=42)

X_train_smote , y_train_smote = smote_nc.fit_resample(X_train_final , y_train)

print("Before SMOTENC:", y_train.value_counts().to_dict())
print("After SMOTENC: ", y_train_smote.value_counts().to_dict())
print("X_train_smote shape:", X_train_smote.shape)

Before SMOTENC: {0: 4139, 1: 1495}
After SMOTENC:  {0: 4139, 1: 4139}
X_train_smote shape: (8278, 30)


In [15]:
print(X_train_smote["contract_One year"].unique())
print(X_train_smote["senior_citizen"].unique())

[1. 0.]
[0 1]


## The alternative: class_weight='balanced'

Rather than creating synthetic data, this approach leaves the training set
untouched and instead penalizes the model more heavily for mistakes on the
minority (churn) class during training. The weight for each class is
computed as `n_samples / (n_classes × count(class))`, which comes out to
roughly 0.68 for retained customers and 1.88 for churned customers —
reflecting the same ~2.77x imbalance ratio in the raw data.

Both this and SMOTENC will be compared directly in the modelling notebook
using AUC-PR and recall, rather than assuming either is better in advance.

In [16]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print(class_weight_dict)

{0: 0.6805991785455424, 1: 1.8842809364548494}


## Saving processed data for the modelling notebook

Two versions of the training data are saved: the encoded-but-unresampled
version (`X_train_encoded`, `y_train`), which is the leakage-safe input for
any pipeline that applies SMOTENC internally per fold, and the manually
resampled version (`X_train_smote_manual`), kept only for quick comparison
or illustration — not used for cross-validation or tuning. The test set
and computed class weights are also saved so the modelling notebook can
load everything without repeating this work.

In [17]:
import joblib

# Leakage-safe path: encoded only, SMOTENC applied inside the pipeline in next steps
joblib.dump(X_train_final, "../data/X_train_encoded.pkl")
joblib.dump(y_train, "../data/y_train.pkl")

# For illustration / quick comparison only — NOT used for CV or tuning
joblib.dump(X_train_smote, "../data/X_train_smote_manual.pkl")
joblib.dump(y_train_smote, "../data/y_train_smote_manual.pkl")

joblib.dump(X_test_final, "../data/X_test.pkl")
joblib.dump(y_test, "../data/y_test.pkl")
joblib.dump(class_weight_dict, "../data/class_weights.pkl")

print("All artifacts saved.")

All artifacts saved.


## Summary: feature engineering decisions

- Dropped `customer_id` (identifier, not a feature).
- Kept `total_charges` despite its correlation with `tenure` (0.83) —
  decision deferred to SHAP analysis in the modelling stage rather than
  dropped on assumption.
- Encoded all categorical columns: binary columns were already 0/1,
  multi-category columns were one-hot encoded with the encoder fit on
  the training set only.
- Split into train/test (80/20, stratified) before any encoding or
  resampling, to avoid data leakage.
- Compared two approaches to the 73.5% / 26.5% class imbalance:
  **SMOTENC** (categorical-safe oversampling) and **class_weight='balanced'**
  (no synthetic data, reweighted loss). Both are carried forward; the real
  comparison happens in the modelling notebook using AUC-PR and recall.
- Resampling will be applied inside an `imblearn.Pipeline` during
  cross-validation and tuning, not as a one-off static step, to keep every
  fold leakage-safe.
- Saved all processed artifacts (`X_train_encoded`, `y_train`,
  `X_train_smote_manual`, `y_train_smote_manual`, `X_test`, `y_test`,
  `class_weights`) for direct use in the next notebook.